In [0]:
from pyspark.sql import functions as F
from delta.tables import DeltaTable 
from datetime import datetime, timezone
datetime.now(timezone.utc)
import uuid 

In [0]:
spark.sql("USE CATALOG novacart_adb")

In [0]:
spark.sql("create schema if not exists bronze_schema")

In [0]:
spark.sql("use schema bronze_schema")

Bronze control table 
--> the latest timestamp already processed 
--> the lates primary key processed at that timestamp 
--> how many rows were written in the latest run 

In [0]:
spark.sql(
    """ 
    create table if not exists ingestion_control(
        layer string , 
        table_name string , 
        ts_col string , 
        pk_col string ,
        last_successful_ts string ,
        last_successful_primary_key bigint , 
        last_run_id string ,
        rows_written bigint , 
        run_status string , 
        updated_at timestamp
    )
    using delta 



    """
)

--Source table configuration 
--this cell defines which source tables  will be loaded into bronze  and which columns should be used as -
1.primary key 
2.timestamp/watermark column
---it also creates unique brinze_run_id

In [0]:

tables_config={

    "orders": {"pk_col": "order_id","ts_col":"updated_at"},
    "products":{"pk_col":"product_id","ts_col":"updated_at"},
    "payments":{"pk_col":"payment_id","ts_col":"processed_at"}
}

bronze_run_id = str(uuid.uuid4())
print("Current  Bronze Run id ", bronze_run_id)

**Helper Functions**
1. get_last_successful_watermark()--> reads the last  processed  watermark from  the control table 
2. upsert_bronze_control()--> updates the control  table after  a successful  bronze  load 

In [0]:
def get_last_successful_watermark(table_name: str):
    ctrl = (
        spark.table("novacart_adb.bronze_schema.ingestion_control")
        .filter(
            (F.col("layer") == "bronze") &
            (F.col("table_name") == table_name) &  # fix here
            (F.col("run_status") == "success")
        )
        .orderBy(F.col("updated_at").desc())
        .limit(1)
    )
    rows = ctrl.collect()

    if not rows:
        return None, None

    return rows[0]["last_successful_ts"], rows[0]["last_successful_primary_key"]

In [0]:
from datetime import datetime
from delta.tables import DeltaTable

def upsert_bronze_control(table_name, ts_col, pk_col,
                           last_ts, last_pk, rows_written, run_id):
    
    now = datetime.utcnow()

    control_df = spark.createDataFrame(
        [(
            "bronze",
            table_name,
            ts_col,
            pk_col,
            last_ts,
            int(last_pk) if last_pk is not None else None,
            run_id,
            int(rows_written),
            "success",
            now
        )],
        schema="""
            layer string,
            table_name string,
            ts_col string,
            pk_col string,
            last_successful_ts timestamp,
            last_successful_primary_key bigint,
            last_run_id string,
            rows_written bigint,
            run_status string,
            updated_at timestamp
        """
    )

    dt = DeltaTable.forName(spark, "novacart_adb.bronze_schema.ingestion_control")

    (dt.alias("t")
     .merge(
         control_df.alias("s"),
         "t.table_name = s.table_name AND t.layer = s.layer"
     )
     .whenMatchedUpdate(set={
         "ts_col":                      "s.ts_col",
         "pk_col":                      "s.pk_col",
         "last_successful_ts":          "s.last_successful_ts",
         "last_successful_primary_key": "s.last_successful_primary_key",
         "last_run_id":                 "s.last_run_id",
         "rows_written":                "s.rows_written",
         "run_status":                  "s.run_status",
         "updated_at":                  "s.updated_at"
     })
     .whenNotMatchedInsertAll()
     .execute()
    )
    
    print(f"  Control table updated for {table_name}")

# **Bronze Incremental load loop **
This is main Bronze logic 
1. reads the latest watermark 
2.reads the source sql table 
3. filters only new/changed rows 
4. adds bronze added columns 
5.Appends the rows  into bronze delta table 
6.updates the control table 
This is core incremental logic 


In [0]:
for table_name, cfg in tables_config.items():
    ts_col = cfg["ts_col"]
    pk_col = cfg["pk_col"]
    source_table = f"novacart_connection_catalog.dbo.{table_name}"
    target_table = f"novacart_adb.bronze_schema.{table_name}_raw"

    last_successful_ts, last_successfull_pk = get_last_successful_watermark(table_name)

    print(f"\n Processing {table_name} table")
    print(f" Last successful ts: {last_successful_ts}")
    print(f" Last successful pk: {last_successfull_pk}")

    source_df = (spark.read.table(source_table)
                 .withColumn(ts_col, F.col(ts_col).cast("timestamp")))

    if last_successful_ts is None:
        rows_to_load = source_df
    else:
        rows_to_load = source_df.filter(
            (F.col(ts_col) > F.lit(last_successful_ts)) |
            (
                (F.col(ts_col) == F.lit(last_successful_ts)) &
                (F.col(pk_col).cast("long") > F.lit(int(last_successfull_pk)))
            )
        )

    rows_to_load = (
        rows_to_load
        .withColumn("bronze_ingested_at", F.current_timestamp())
        .withColumn("bronze_run_id", F.lit(bronze_run_id))
        .withColumn("bronze_source_table", F.lit(source_table))
    )

    rows_count = rows_to_load.count()
    print(f" {table_name} rows to load: {rows_count}")

    if rows_count == 0:
        print(f" No new rows for {table_name}.")
        upsert_bronze_control(
            table_name,
            ts_col,
            pk_col,
            last_successful_ts,
            last_successfull_pk,
            rows_count,
            bronze_run_id
        )
        continue

    # write to bronze Delta table
    rows_to_load.write.format("delta").mode("append").saveAsTable(target_table)

    # get max timestamp from this batch
    max_ts = (rows_to_load
              .agg(F.max(ts_col).alias("max_ts"))
              .collect()[0]["max_ts"])

    # get max pk at that timestamp
    max_pk = (rows_to_load
              .filter(F.col(ts_col) == F.lit(max_ts))
              .agg(F.max(F.col(pk_col).cast("long")).alias("max_pk"))
              .collect()[0]["max_pk"])

    upsert_bronze_control(
        table_name,
        ts_col,
        pk_col,
        max_ts,
        max_pk,
        rows_count,
        bronze_run_id
    )

    print(f" Wrote {rows_count} rows to {target_table}")

In [0]:
print("Orders  Bronze count:",spark.sql("select count(*) from novacart_adb.bronze_schema.orders_raw ").collect()[0][0])
print("Products Bronze count",spark.sql("select count(*) from novacart_adb.bronze_schema.products_raw ").collect()[0][0])
print("Payments Bronze count",spark.sql("select count(*) from novacart_adb.bronze_schema.payments_raw ").collect()[0][0])
display(spark.sql("select * from novacart_adb.bronze_schema.orders_raw"))
display(spark.sql("select * from novacart_adb.bronze_schema.products_raw"))
display(spark.sql("select * from novacart_adb.bronze_schema.payments_raw"))
display(spark.sql("select * from novacart_adb.bronze_schema.ingestion_control ").orderBy("table_name"))